# 09 — Production Guardrails: Guard Any LLM SDK in 3 Lines

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anulum/director-ai/blob/main/notebooks/09_production_guardrails.ipynb)

This tutorial covers the `guard()` wrapper — the fastest way to add
hallucination detection to **any** LLM SDK. Supports OpenAI, Anthropic,
Bedrock, Gemini, and Cohere out of the box.

**What you'll learn:**
1. Wrap an OpenAI client with `guard()` — 3 lines
2. Wrap Anthropic, Bedrock, Gemini, Cohere clients
3. Choose failure modes: `raise`, `log`, or `metadata`
4. Guard streaming responses
5. Add domain-specific ground truth
6. Production patterns: async, retry, graceful degradation

In [ ]:
!pip install -q director-ai

## 1. The `guard()` One-Liner

`guard()` wraps any supported SDK client. It intercepts every
`.create()` call, scores the response, and acts on low coherence.

```python
from openai import OpenAI
from director_ai import guard

client = guard(OpenAI(), facts={"refund": "Refunds within 30 days only."})
# Now use client.chat.completions.create() as usual — auto-scored.
```

Since we don't need an API key for this tutorial, we'll mock the SDK shape.

In [ ]:
from unittest.mock import MagicMock
from director_ai import guard, get_score, HallucinationError


def make_mock_openai(response_text: str):
    """Build a mock that looks like openai.OpenAI() to guard()."""
    client = MagicMock()
    choice = MagicMock()
    choice.message.content = response_text
    client.chat.completions.create.return_value = MagicMock(choices=[choice])
    return client


# Accurate response — passes guardrail
client = make_mock_openai("Refunds are available within 30 days.")
client = guard(
    client,
    facts={"refund_policy": "Refunds within 30 days only."},
    on_fail="metadata",
)

resp = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "What is your refund policy?"}],
)

score = get_score()
print(f"Response: {resp.choices[0].message.content}")
print(f"Coherence: {score.score:.3f}")
print(f"Approved: {score.approved}")

## 2. Failure Modes

| Mode | Behaviour |
|------|-----------|
| `on_fail="raise"` | Raises `HallucinationError` — caller handles it |
| `on_fail="log"` | Logs warning, returns response unchanged |
| `on_fail="metadata"` | Stores score in context var, retrieve via `get_score()` |

In [ ]:
# on_fail="raise" — catches hallucination
client_bad = make_mock_openai("We offer lifetime unlimited refunds.")
client_bad = guard(
    client_bad,
    facts={"refund_policy": "Refunds within 30 days only. No exceptions."},
    on_fail="raise",
)

try:
    client_bad.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": "What is your refund policy?"}],
    )
    print("Passed (no hallucination detected)")
except HallucinationError as e:
    print(f"Caught hallucination: coherence={e.score.score:.3f}")
    print(f"Response was: {e.response_text[:80]}")

## 3. All Supported SDKs

`guard()` detects the SDK shape automatically:

```python
# OpenAI / vLLM / Groq / LiteLLM / Together / Ollama
client = guard(OpenAI(), facts=facts)

# Anthropic
client = guard(Anthropic(), facts=facts)

# AWS Bedrock
client = guard(boto3.client("bedrock-runtime"), facts=facts)

# Google Gemini
client = guard(genai.GenerativeModel("gemini-pro"), facts=facts)

# Cohere
client = guard(cohere.Client(), facts=facts)
```

Each wraps the relevant method (`.create()`, `.converse()`, `.generate_content()`, `.chat()`) transparently.

In [ ]:
# Anthropic mock
def make_mock_anthropic(response_text: str):
    client = MagicMock(spec_set=["messages"])  # no .chat attr
    client.chat = None  # force Anthropic path
    del client.chat  # remove it entirely
    content_block = MagicMock()
    content_block.text = response_text
    client.messages.create.return_value = MagicMock(content=[content_block])
    return client


anthropic_client = make_mock_anthropic("Paris is the capital of France.")
anthropic_client = guard(
    anthropic_client,
    facts={"capital": "Paris is the capital of France."},
    on_fail="metadata",
)

resp = anthropic_client.messages.create(
    model="claude-sonnet-4-6",
    messages=[{"role": "user", "content": "What is the capital of France?"}],
    max_tokens=100,
)

score = get_score()
print(f"Anthropic response scored: {score.score:.3f}, approved={score.approved}")

## 4. Guarding Streams

When `stream=True`, `guard()` performs periodic coherence checks
every 8 tokens and a final check when the stream completes.

```python
client = guard(OpenAI(), facts=facts, on_fail="raise")

stream = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "Explain refund policy"}],
    stream=True,
)

for chunk in stream:  # raises mid-stream if hallucination detected
    print(chunk.choices[0].delta.content, end="")
```

## 5. Production Pattern: Graceful Degradation

In production, you want the guardrail to degrade gracefully — never
block the user if the scorer itself fails.

In [ ]:
import logging


def guarded_chat(client, messages, facts):
    """Production wrapper: guard with graceful fallback."""
    try:
        guarded = guard(client, facts=facts, on_fail="metadata")
        resp = guarded.chat.completions.create(
            model="gpt-4o", messages=messages
        )
        score = get_score()

        if score and not score.approved:
            logging.warning(
                "Low coherence %.3f — flagging for review", score.score
            )
            # Option A: Return with warning metadata
            # Option B: Retry with different temperature
            # Option C: Escalate to human

        return resp, score

    except Exception:
        logging.exception("Guard failed — falling back to unguarded")
        # Fall back to raw client
        return client.chat.completions.create(
            model="gpt-4o", messages=messages
        ), None


# Demo
client = make_mock_openai("Our refund window is 30 days.")
resp, score = guarded_chat(
    client,
    [{"role": "user", "content": "What is the refund policy?"}],
    {"policy": "30-day refund window."},
)
print(f"Score: {score.score:.3f}" if score else "Score: N/A (fallback)")

## 6. Custom Ground Truth from Files

For production, load facts from your knowledge base rather than inline dicts.

In [ ]:
from director_ai.core import GroundTruthStore

store = GroundTruthStore()

# Load from a dict (typical: read from DB, JSON, or YAML)
knowledge_base = {
    "pricing": "Basic plan $9/mo, Pro $29/mo, Enterprise custom.",
    "uptime": "99.9% SLA for Pro and Enterprise tiers.",
    "support": "Email support for Basic, 24/7 chat for Pro+.",
    "data_retention": "90 days for Basic, unlimited for Pro+.",
    "gdpr": "Fully GDPR compliant. Data stored in EU (Frankfurt).",
}

for key, fact in knowledge_base.items():
    store.add(key, fact)

print(f"Loaded {len(store.facts)} facts into ground truth store")

# Use with guard()
client = make_mock_openai("The Basic plan costs $9 per month.")
client = guard(client, store=store, on_fail="metadata")

resp = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "How much is the Basic plan?"}],
)
score = get_score()
print(f"Coherence: {score.score:.3f}, approved={score.approved}")

## Summary

| Feature | Code |
|---------|------|
| Guard OpenAI | `guard(OpenAI(), facts=facts)` |
| Guard Anthropic | `guard(Anthropic(), facts=facts)` |
| Guard Bedrock | `guard(boto3.client('bedrock-runtime'), facts=facts)` |
| Guard Gemini | `guard(genai.GenerativeModel('gemini-pro'), facts=facts)` |
| Guard Cohere | `guard(cohere.Client(), facts=facts)` |
| Raise on fail | `on_fail="raise"` |
| Log on fail | `on_fail="log"` |
| Score metadata | `on_fail="metadata"` + `get_score()` |
| Custom threshold | `guard(client, threshold=0.7)` |
| NLI model | `guard(client, use_nli=True)` |